# RAG over My Notes 🗂️💬
Ask questions about **GDPR** or **ML** and get answers grounded in *your own vault*, with links to the exact note and section. Along the way you build every piece of a retrieval-augmented generation system: chunking, keyword search, embeddings, hybrid fusion, evaluation and the LLM prompt.

Guide note: [[RAG over My Notes]] · background: [[RAG and Agents]], [[Text Representations]], [[LLMs Overview]]

**Does it need lots of notes?** No. RAG doesn't *train* on your notes, it *searches* them, so even 10 notes work. New IT-ret lectures are picked up the next time you run the notebook. Embeddings are cached, so only new or changed sections get embedded.

**Needs:** `pip install sentence-transformers` (multilingual embeddings, ~470 MB model download the first time). For generated answers, also `pip install anthropic` and set `ANTHROPIC_API_KEY`. Without a key you still get the retrieved sections with links.

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import hashlib, os, pathlib, re, urllib.parse, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import Markdown, display
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

VAULT = pathlib.Path("../..").resolve()
VAULT_NAME = VAULT.name                                   # used for obsidian:// links
FOLDERS = ["Machine Learning", "9-semester/IT-lov"]      # add course folders here
SKIP = ("Flashcards",)
files = [f for d in FOLDERS for f in sorted((VAULT / d).rglob("*.md")) if not any(s in f.parts for s in SKIP)]
print(f"{len(files)} notes in {FOLDERS}")

## 1. Chunking by headings
Whole notes are too long to retrieve (and to fit in a prompt); fixed-size windows cut sentences in half. Obsidian notes have **headings**, so use them: one chunk per section (`#`, `##`, `###`). Prefix every chunk with its *breadcrumb* (`Note > Heading > Subheading`) so a chunk saying "it is computed as…" still knows what *it* is.

Return a list of dicts `{"note", "heading", "text"}`: strip YAML frontmatter, skip sections whose body is shorter than `min_chars`, and split bodies longer than `max_chars` into several chunks (each with the breadcrumb).

In [ ]:
def chunk_markdown(text, title, max_chars=1500, min_chars=40):
    # TODO 1: split on #/##/### headings, keep a heading path, prefix "Title > H1 > H2\n", skip short, split long
    raise NotImplementedError  # TODO 1

In [ ]:
TOY = """---
tags: [x]
---
# GDPR
Intro text that is long enough to keep as its own chunk here.
## Art. 6
Lawful bases: consent, contract, legal obligation, vital interests, public task, legitimate interests.
## Tiny
too short
"""
def _t_chunk():
    c = chunk_markdown(TOY, "Lektion 3")
    long = chunk_markdown("# A\n" + "x" * 3100, "N", max_chars=1500)
    return ([x["heading"] for x in c] == ["GDPR", "Art. 6"] and c[1]["text"].startswith("Lektion 3 > GDPR > Art. 6\n")
            and "tags" not in c[0]["text"] and len(long) == 3 and all(x["text"].startswith("N > A\n") for x in long))
check("chunking", _t_chunk, "Expect 2 chunks (GDPR, Art. 6 – 'Tiny' is too short), breadcrumb 'Lektion 3 > GDPR > Art. 6', no frontmatter; 3100 chars → 3 chunks.")

In [ ]:
if ready("chunking"):
    chunks = [dict(c, path=str(f.relative_to(VAULT))) for f in files for c in chunk_markdown(f.read_text(encoding="utf-8"), f.stem)]
    texts = [c["text"] for c in chunks]
    print(f"{len(chunks):,} chunks, median {int(np.median([len(t) for t in texts]))} characters")
    print(chunks[0]["text"][:300])

## 2. Two retrievers
- **TF-IDF** (keyword search): great for exact terms like *art. 6*, *DPIA*, *LoRA*; useless for synonyms.
- **Dense embeddings** with [`multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small): a sentence encoder trained so a question and a passage that *answers* it get similar vectors, across ~100 languages, so an English question can find a Danish note. (e5 expects the prefixes `query: ` and `passage: `.)

In [ ]:
if ready("chunking"):
    tfidf = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2)); X_tfidf = tfidf.fit_transform(texts)
    def search_tfidf(q, n=50): return list(np.argsort(-(X_tfidf @ tfidf.transform([q]).T).toarray().ravel())[:n])

USE_DENSE = True
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    USE_DENSE = False; print("⚠️  sentence-transformers not installed → keyword search only (pip install sentence-transformers)")

CACHE = pathlib.Path("data/rag_embeddings.npz"); CACHE.parent.mkdir(exist_ok=True)
if USE_DENSE and ready("chunking"):
    encoder = SentenceTransformer("intfloat/multilingual-e5-small")
    key = lambda t: hashlib.sha1(t.encode()).hexdigest()
    cache = dict(np.load(CACHE)) if CACHE.exists() else {}
    todo = [t for t in texts if key(t) not in cache]
    print(f"embedding {len(todo)} new/changed chunks ({len(texts) - len(todo)} cached)… (first run ≈ 2 min on a CPU)")
    if todo:
        for t, e in zip(todo, encoder.encode(["passage: " + t for t in todo], batch_size=64, normalize_embeddings=True)):
            cache[key(t)] = e
        np.savez(CACHE, **{k: cache[k] for k in {key(t) for t in texts}})      # drop embeddings of deleted sections
    E = np.stack([cache[key(t)] for t in texts])
    def search_dense(q, n=50): return list(np.argsort(-(E @ encoder.encode("query: " + q, normalize_embeddings=True)))[:n])

## 3. Hybrid search with reciprocal rank fusion
The two retrievers fail on different questions, so combine them. Scores aren't comparable (TF-IDF cosine vs embedding cosine), but **ranks** are. Reciprocal rank fusion gives each document $\sum_{\text{lists}} \frac{1}{k + \text{rank} + 1}$ (rank starting at 0, $k = 60$) and sorts by that. Return the fused list of document ids, best first.

In [ ]:
def rrf(rankings, k=60):
    # TODO 2: fuse several ranked lists of doc ids into one ranked list
    raise NotImplementedError  # TODO 2

In [ ]:
check("RRF", lambda: rrf([["a", "b", "c"], ["c", "a", "d"]]) == ["a", "c", "b", "d"] and rrf([[3, 1]], k=0) == [3, 1],
      "a: 1/61 + 1/62, c: 1/63 + 1/61, b: 1/62, d: 1/63 → [a, c, b, d].")

def search(q, n=8, mode="hybrid"):
    if mode == "tfidf" or not USE_DENSE: return search_tfidf(q)[:n]
    if mode == "dense": return search_dense(q)[:n]
    return rrf([search_tfidf(q), search_dense(q)])[:n]

def notes_of(ids, k):   # distinct notes in rank order
    out = []
    for i in ids:
        if chunks[i]["note"] not in out: out.append(chunks[i]["note"])
    return out[:k]

## 4. Measure it: does the right note come back?
Retrieval quality is the whole game: the LLM can't cite what it never sees. A small hand-made **eval set** of (question, acceptable notes) pairs, mixing Danish GDPR and English ML, lets you compare retrievers and catch regressions when you change chunking. *Hit@k* = the fraction of questions where an acceptable note is among the top-k distinct notes.

👉 Add your own questions as you study; it doubles as a revision list.

In [ ]:
EVAL = [
    ("Hvornår skal man lave en konsekvensanalyse (DPIA)?", ["Lektion 4 - Sikkerhed, konsekvensanalyse, brud, DPO og tilsyn", "GDPR-tjekliste til synopsis og eksamen"]),
    ("Hvilke behandlingsgrundlag findes i artikel 6?", ["Lektion 3 - Behandlingsgrundlag og de registreredes rettigheder"]),
    ("Hvad er retten til at blive glemt?", ["Lektion 3 - Behandlingsgrundlag og de registreredes rettigheder"]),
    ("Hvornår skal et brud på persondatasikkerheden anmeldes til Datatilsynet?", ["Lektion 4 - Sikkerhed, konsekvensanalyse, brud, DPO og tilsyn"]),
    ("Hvad er hacking efter straffeloven § 263?", ["Lektion 5 - Cybercrime"]),
    ("Hvad er GDPR's territoriale anvendelsesområde?", ["Lektion 1 - GDPR intro, anvendelsesområde og begreber"]),
    ("Hvad er forskellen på en dataansvarlig og en databehandler?", ["Lektion 2 - Aktører og behandlingsprincipper", "Begrebsliste IT-ret"]),
    ("What is the difference between a data controller and a data processor?", ["Lektion 2 - Aktører og behandlingsprincipper", "Begrebsliste IT-ret"]),
    ("What counts as personal data under GDPR?", ["Lektion 1 - GDPR intro, anvendelsesområde og begreber", "Begrebsliste IT-ret"]),
    ("Why do we scale dot-product attention by the square root of d_k?", ["Attention Mechanism", "Transformers"]),
    ("How does dropout prevent overfitting?", ["Overfitting and Regularization", "Training Tricks"]),
    ("What is the difference between bagging and boosting?", ["Ensemble Methods"]),
    ("How does k-means choose its clusters?", ["Clustering"]),
    ("What is the bias-variance tradeoff?", ["Bias-Variance Tradeoff"]),
    ("How does the BPR loss work for implicit feedback?", ["Collaborative Filtering and Matrix Factorization", "Project - MovieLens Recommender"]),
    ("What is the Bellman equation?", ["RL Basics and MDPs"]),
    ("How does backpropagation compute gradients?", ["Backpropagation"]),
    ("Why do gradients vanish in an RNN?", ["RNN and LSTM"]),
    ("How should I cross-validate a forecasting model?", ["Time Series Validation and Features", "Time Series Forecasting"]),
    ("How does LoRA fine-tuning work?", ["Fine-tuning and Alignment"]),
]

def hit_at_k(retrieve_notes, eval_set, k=3):
    # TODO 4: fraction of (question, gold_notes) where any gold note is in retrieve_notes(question, k)
    raise NotImplementedError  # TODO 4

In [ ]:
check("hit@k", lambda: hit_at_k(lambda q, k: ["A", "B", "C"][:k], [("q1", ["B"]), ("q2", ["C", "Z"]), ("q3", ["Z"])], k=2) == 1 / 3,
      "Toy: top-2 = [A, B] → q1 hit, q2 miss (C is 3rd), q3 miss → 1/3.")

scores = {}
if ready("chunking", "RRF", "hit@k"):
    known = {c["note"] for c in chunks}; missing = {g for _, gs in EVAL for g in gs} - known
    if missing: print("⚠️  eval notes not in the corpus:", missing)
    for mode in (["tfidf", "dense", "hybrid"] if USE_DENSE else ["tfidf"]):
        scores[mode] = {k: hit_at_k(lambda q, k: notes_of(search(q, 30, mode), k), EVAL, k) for k in (1, 3)}
    print(pd.DataFrame(scores).T.rename(columns=lambda k: f"hit@{k}").round(2))

In [ ]:
if ready("chunking", "RRF", "hit@k") and USE_DENSE:
    check("hybrid retrieval works", lambda: scores["hybrid"][3] >= 0.8 and scores["hybrid"][3] >= scores["tfidf"][3],
          "Hybrid hit@3 should be ≥ 0.8 and at least as good as TF-IDF alone.")
    for q, gold in EVAL:
        got = notes_of(search(q, 30), 3)
        if not any(g in got for g in gold): print(f"miss: {q}\n      wanted {gold[0]!r}, got {got}")

Look at the misses: English questions about GDPR tend to land on English *ML* notes about data, because the cross-lingual signal is weaker than the same-language one. Fixes to try: translate the query (ask the LLM to rewrite it in Danish), add English keywords to your Danish notes, or filter by folder when the topic is clear.

## 5. The prompt
The LLM sees numbered sources and must answer **only** from them, citing `[n]`. Build the user message: the sources as `[1] <chunk text>`, `[2] …` (numbering from 1), then the question, then the instructions (answer only from the sources, cite like `[1]`, answer in the question's language, say so if the answer isn't there).

In [ ]:
def build_prompt(question, hits):
    # TODO 3: numbered sources "[1] text", the question, and the grounding/citation instructions
    raise NotImplementedError  # TODO 3

In [ ]:
def _t_prompt():
    p = build_prompt("Hvad er art. 6?", [{"text": "Lektion 3 > Art. 6\nSeks grundlag"}, {"text": "Begrebsliste > Samtykke\nFrivilligt"}])
    return "[1] Lektion 3 > Art. 6" in p and "[2] Begrebsliste" in p and "Hvad er art. 6?" in p and "[3]" not in p.split("Question")[0]
check("prompt", _t_prompt, "The prompt must contain '[1] <first chunk>', '[2] <second chunk>' and the question.")

## 6. Ask your notes 💬
`ask()` retrieves the top sections, sends the prompt to Claude (if `ANTHROPIC_API_KEY` is set) and prints the answer with **clickable links** to the sources. `obsidian://` links open the note in Obsidian; the `[[Note#Heading]]` form is ready to paste into a note. A Sonnet-class model is plenty for this and cheap; change `MODEL` if you like.

In [ ]:
MODEL = "claude-sonnet-5-5"
SYSTEM = "You are a study assistant for a computer science master's student. Be concise and precise."
try:
    import anthropic
    client = anthropic.Anthropic() if os.environ.get("ANTHROPIC_API_KEY") else None
except ImportError:
    client = None

def obsidian_link(c):
    return "obsidian://open?" + urllib.parse.urlencode({"vault": VAULT_NAME, "file": c["path"][:-3]}, quote_via=urllib.parse.quote)

def ask(question, n=5, show_prompt=False):
    hits = [chunks[i] for i in search(question, n)]
    prompt = build_prompt(question, hits)
    if show_prompt: print(prompt)
    if client:
        resp = client.messages.create(model=MODEL, max_tokens=700, system=SYSTEM, messages=[{"role": "user", "content": prompt}])
        answer = resp.content[0].text
    else:
        answer = "_(No `ANTHROPIC_API_KEY` set: showing the retrieved sources only. They are what the LLM would read.)_"
    refs = "\n".join(f"{k}. [{c['note']} › {c['heading'] or 'top'}]({obsidian_link(c)}) · `[[{c['note']}#{c['heading']}]]`" for k, c in enumerate(hits, 1))
    display(Markdown(f"**Q: {question}**\n\n{answer}\n\n**Sources**\n{refs}"))
    return answer

if ready("chunking", "RRF", "prompt"):
    ask("Hvornår skal man udpege en DPO?")
    ask("Why does LightGCN drop the nonlinearity?")

<details><summary>▶ Solution</summary>

```python
def chunk_markdown(text, title, max_chars=1500, min_chars=40):
    text = re.sub(r"^---\n.*?\n---\n", "", text, flags=re.S)
    chunks, path, buf = [], [], []
    def flush():
        body = "\n".join(buf).strip()
        if len(body) < min_chars:
            return
        crumb = " > ".join([title] + path)
        for start in range(0, len(body), max_chars):
            chunks.append({"note": title, "heading": path[-1] if path else "", "text": crumb + "\n" + body[start:start + max_chars]})
    for line in text.splitlines():
        m = re.match(r"^(#{1,3})\s+(.*)", line)
        if m:
            flush(); buf = []
            level = len(m.group(1)); path = path[:level - 1] + [m.group(2).strip()]
        else:
            buf.append(line)
    flush()
    return chunks
```

```python
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=lambda d: -scores[d])
```

```python
def build_prompt(question, hits):
    sources = "\n\n".join(f"[{n}] {c['text']}" for n, c in enumerate(hits, 1))
    return (f"Sources from my notes:\n\n{sources}\n\n"
            f"Question: {question}\n\n"
            "Answer using ONLY the sources above and cite them like [1] or [2][3]. "
            "Answer in the same language as the question. If the sources don't contain the answer, say so.")
```

```python
def hit_at_k(retrieve_notes, eval_set, k=3):
    hits = []
    for question, gold in eval_set:
        notes = retrieve_notes(question, k)
        hits.append(any(g in notes for g in gold))
    return float(np.mean(hits))
```
</details>

## Make it yours (stretch goals)
- **Quiz mode:** pick a random chunk from an IT-ret note and ask the LLM to write an exam-style question about it; answer it yourself; let the LLM grade you against the chunk.
- **Re-ranking:** retrieve 30 with hybrid search, then re-rank with a cross-encoder (e.g. `BAAI/bge-reranker-v2-m3`, multilingual) and measure hit@1 again.
- **Query rewriting:** ask the LLM to rewrite an English GDPR question into Danish legal terms before searching. Does hit@3 on the English questions go up?
- **Chunk size study:** try `max_chars` 500 / 1500 / 4000 and plot hit@3. Smaller = more precise but less context.
- **Answer faithfulness:** for 10 questions, check by hand whether every cited `[n]` actually supports its sentence. That's the metric that matters for exam prep.
- **A tiny app:** wrap `ask()` in a [Gradio](https://www.gradio.app/) or Streamlit UI and keep it open while studying.